# MLOps Task 2 — Notebook 1: Read and Join the Tables

## Objective

The objective of this notebook is to read the Olist tables from the local PostgreSQL database, inspect their structure and relationships, and create a machine learning table with one row per order.

Tables with multiple rows per order will be aggregated before joining to prevent duplicate orders in the final dataset.

The output of this notebook will be saved as an artifact and used as the input for Notebook 2.

## 1. Imports and Database Connection

This notebook uses Pandas to work with the data and SQLAlchemy to connect to the local PostgreSQL database created in Task 1.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

In [ ]:
# PostgreSQL database connection
DB_USER = "postgres"
DB_PASSWORD = "PASSWORD"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist"

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database connection engine created successfully.")

Database connection engine created successfully.


## 2. Test the Database Connection

Before reading the Olist tables, the connection to the local PostgreSQL database is tested to confirm that the database is accessible.

In [3]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Database connection successful:", result.scalar())

Database connection successful: 1


## 3. Read and Inspect the Olist Tables

Before joining the tables, each table is inspected individually to understand its structure, row count, key columns, duplicate records, and the meaning of one row.

This inspection helps identify one-to-one and one-to-many relationships and prevents incorrect joins that could duplicate orders in the final machine learning table.

In [4]:
# List of Olist tables stored in PostgreSQL
table_names = [
    "olist_customers",
    "olist_orders",
    "olist_sellers",
    "product_category_name_translation",
    "olist_products",
    "olist_order_items",
    "olist_order_payments",
    "olist_order_reviews",
    "olist_geolocation"
]

# Read each table from PostgreSQL
tables = {}

for table_name in table_names:
    tables[table_name] = pd.read_sql(
        f"SELECT * FROM {table_name}",
        engine
    )

print("All tables loaded successfully!")

All tables loaded successfully!


In [5]:
for table_name, df in tables.items():
    print(f"{table_name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

olist_customers: 99,441 rows, 5 columns
olist_orders: 99,441 rows, 8 columns
olist_sellers: 3,095 rows, 4 columns
product_category_name_translation: 71 rows, 2 columns
olist_products: 32,951 rows, 9 columns
olist_order_items: 112,650 rows, 7 columns
olist_order_payments: 103,886 rows, 5 columns
olist_order_reviews: 99,224 rows, 7 columns
olist_geolocation: 1,000,163 rows, 5 columns


## 4. Table Structure and Key Inspection

Each table is inspected individually to identify its key columns, check for duplicate records, and understand what one row represents.

This information is used to determine how the tables should be aggregated and joined while preserving one row per order in the final machine learning table.

In [6]:
# Define the main key or relationship columns for each table
key_columns = {
    "olist_customers": ["customer_id"],
    "olist_orders": ["order_id", "customer_id"],
    "olist_sellers": ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
    "olist_products": ["product_id"],
    "olist_order_items": ["order_id", "order_item_id"],
    "olist_order_payments": ["order_id", "payment_sequential"],
    "olist_order_reviews": ["review_id", "order_id"],
    "olist_geolocation": ["geolocation_zip_code_prefix"]
}

# Inspect each table
for table_name, df in tables.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")

    print("\nKey column duplicate check:")
    for column in key_columns[table_name]:
        print(f"  {column}: {df[column].duplicated().sum()} duplicate values")

    print("\nFirst 3 rows:")
    display(df.head(3))


TABLE: olist_customers
Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Exact duplicate rows: 0

Key column duplicate check:
  customer_id: 0 duplicate values

First 3 rows:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



TABLE: olist_orders
Shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Exact duplicate rows: 0

Key column duplicate check:
  order_id: 0 duplicate values
  customer_id: 0 duplicate values

First 3 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



TABLE: olist_sellers
Shape: (3095, 4)
Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
Exact duplicate rows: 0

Key column duplicate check:
  seller_id: 0 duplicate values

First 3 rows:


...

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71



TABLE: olist_order_reviews
Shape: (99224, 7)
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
Exact duplicate rows: 0

Key column duplicate check:
  review_id: 814 duplicate values
  order_id: 551 duplicate values

First 3 rows:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24



TABLE: olist_geolocation
Shape: (1000163, 5)
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
Exact duplicate rows: 261831

Key column duplicate check:
  geolocation_zip_code_prefix: 981148 duplicate values

First 3 rows:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


## 5. Table Relationship Summary

The inspection shows that the Olist dataset contains tables with different levels of granularity. The final machine learning table must contain one row per order, so tables with multiple rows per order must be aggregated before joining.

| Table | Main Key | What One Row Represents | Join Strategy |
|---|---|---|---|
| `olist_orders` | `order_id` | One order | Base table |
| `olist_customers` | `customer_id` | One customer record associated with an order | Join directly |
| `olist_order_items` | `order_id` + `order_item_id` | One item within an order | Aggregate to one row per order |
| `olist_order_payments` | `order_id` + `payment_sequential` | One payment record within an order | Aggregate to one row per order |
| `olist_products` | `product_id` | One product | Join to order items before aggregation |
| `olist_sellers` | `seller_id` | One seller | Join to order items before aggregation |
| `product_category_name_translation` | `product_category_name` | One category translation | Join to products |
| `olist_order_reviews` | Not strictly one row per `order_id` | One review record | Aggregate or select carefully before joining |
| `olist_geolocation` | No unique ZIP code key | One geographic observation | Aggregate by ZIP code before joining |

In [7]:
# Check the number of unique orders in each order-related table

order_relationship_check = pd.DataFrame({
    "table": [
        "olist_orders",
        "olist_order_items",
        "olist_order_payments",
        "olist_order_reviews"
    ],
    "total_rows": [
        len(tables["olist_orders"]),
        len(tables["olist_order_items"]),
        len(tables["olist_order_payments"]),
        len(tables["olist_order_reviews"])
    ],
    "unique_order_ids": [
        tables["olist_orders"]["order_id"].nunique(),
        tables["olist_order_items"]["order_id"].nunique(),
        tables["olist_order_payments"]["order_id"].nunique(),
        tables["olist_order_reviews"]["order_id"].nunique()
    ]
})

order_relationship_check["extra_rows_due_to_multiple_records"] = (
    order_relationship_check["total_rows"]
    - order_relationship_check["unique_order_ids"]
)

order_relationship_check

,table,total_rows,unique_order_ids,extra_rows_due_to_multiple_records
0,olist_orders,99441,99441,0
1,olist_order_items,112650,98666,13984
2,olist_order_payments,103886,99440,4446
3,olist_order_reviews,99224,98673,551


## 6. Prepare Order-Level Item Features

The `olist_order_items` table contains multiple rows for some orders because one order can contain multiple items.

Product and seller information is first joined to the item-level data. The resulting information is then aggregated by `order_id` so that the final dataset continues to contain one row per order.

In [8]:
# Create shorter DataFrame names
orders = tables["olist_orders"]
customers = tables["olist_customers"]
sellers = tables["olist_sellers"]
products = tables["olist_products"]
order_items = tables["olist_order_items"]
payments = tables["olist_order_payments"]
reviews = tables["olist_order_reviews"]
geolocation = tables["olist_geolocation"]
categories = tables["product_category_name_translation"]

In [11]:
# Join products to order items
items_enriched = order_items.merge(
    products,
    on="product_id",
    how="left"
)

# Join English product category names
items_enriched = items_enriched.merge(
    categories,
    on="product_category_name",
    how="left"
)

# Join seller information
items_enriched = items_enriched.merge(
    sellers,
    on="seller_id",
    how="left"
)

print("Enriched order items shape:", items_enriched.shape)
print("Unique orders:", items_enriched["order_id"].nunique())

Enriched order items shape: (112650, 19)
Unique orders: 98666


### 6.1 Aggregate Item-Level Data

After enriching the order items with product and seller information, item-level records are aggregated by `order_id`.

The aggregation creates order-level summary information while preserving the one-row-per-order structure required for the machine learning table.

In [12]:
# Aggregate item-level information to one row per order

order_items_agg = items_enriched.groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    unique_products=("product_id", "nunique"),
    unique_sellers=("seller_id", "nunique"),
    
    total_item_price=("price", "sum"),
    mean_item_price=("price", "mean"),
    max_item_price=("price", "max"),
    
    total_freight_value=("freight_value", "sum"),
    mean_freight_value=("freight_value", "mean"),
    
    mean_product_weight_g=("product_weight_g", "mean"),
    
    product_category_count=(
        "product_category_name",
        "nunique"
    ),
    
    seller_state_count=("seller_state", "nunique")
).reset_index()

print("Aggregated order items shape:", order_items_agg.shape)
print("Unique orders:", order_items_agg["order_id"].nunique())

display(order_items_agg.head())

Aggregated order items shape: (98666, 12)
Unique orders: 98666


,order_id,item_count,unique_products,unique_sellers,total_item_price,mean_item_price,max_item_price,total_freight_value,mean_freight_value,mean_product_weight_g,product_category_count,seller_state_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,58.90,58.90,13.29,13.29,650.0,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,239.90,239.90,19.93,19.93,30000.0,1,1
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,199.00,199.00,17.87,17.87,3050.0,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.99,12.99,12.79,12.79,200.0,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,199.90,199.90,18.14,18.14,3750.0,1,1


### 6.2 Aggregate Payment-Level Data

The `olist_order_payments` table can contain multiple payment records for a single order.

The payment records are therefore aggregated by `order_id` before being joined with the order-level data. This prevents multiple payment rows from creating duplicate orders in the final machine learning table.

In [13]:
# Aggregate payment information to one row per order

payments_agg = payments.groupby("order_id").agg(
    payment_count=("payment_sequential", "count"),
    total_payment_value=("payment_value", "sum"),
    mean_payment_value=("payment_value", "mean"),
    max_payment_installments=("payment_installments", "max")
).reset_index()

print("Aggregated payments shape:", payments_agg.shape)
print("Unique orders:", payments_agg["order_id"].nunique())

display(payments_agg.head())

Aggregated payments shape: (99440, 5)
Unique orders: 99440


,order_id,payment_count,total_payment_value,mean_payment_value,max_payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,218.04,3


### 6.3 Aggregate Review-Level Data

The `olist_order_reviews` table contains multiple review records for some orders.

To preserve one row per order, review records are aggregated by `order_id`. The aggregation keeps the number of reviews and summary statistics for the review score.

In [14]:
# Aggregate review information to one row per order

reviews_agg = reviews.groupby("order_id").agg(
    review_count=("review_id", "count"),
    mean_review_score=("review_score", "mean"),
    min_review_score=("review_score", "min"),
    max_review_score=("review_score", "max")
).reset_index()

print("Aggregated reviews shape:", reviews_agg.shape)
print("Unique orders:", reviews_agg["order_id"].nunique())

display(reviews_agg.head())

Aggregated reviews shape: (98673, 5)
Unique orders: 98673


,order_id,review_count,mean_review_score,min_review_score,max_review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,5,5
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,4,4
2,000229ec398224ef6ca0657da4fc703e,1,5.0,5,5
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,4,4
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,5,5


### 6.4 Aggregate Geolocation Data

The geolocation table contains multiple geographic observations for the same ZIP-code prefix.

Because the final machine learning table must contain one row per order, the geolocation data is aggregated to one record per ZIP-code prefix before it is joined to customer or seller information.

In [15]:
# Aggregate geolocation data to one row per ZIP-code prefix

geolocation_agg = geolocation.groupby(
    "geolocation_zip_code_prefix"
).agg(
    latitude=("geolocation_lat", "mean"),
    longitude=("geolocation_lng", "mean"),
    city=("geolocation_city", "first"),
    state=("geolocation_state", "first")
).reset_index()

print("Aggregated geolocation shape:", geolocation_agg.shape)
print(
    "Unique ZIP-code prefixes:",
    geolocation_agg["geolocation_zip_code_prefix"].nunique()
)

display(geolocation_agg.head())

Aggregated geolocation shape: (19015, 5)
Unique ZIP-code prefixes: 19015


,geolocation_zip_code_prefix,latitude,longitude,city,state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


In [16]:
# Merge customer information with geolocation data
customers_enriched = customers.merge(
    geolocation_agg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

print("Customers enriched shape:", customers_enriched.shape)
print("Unique customers:", customers_enriched["customer_id"].nunique())

Customers enriched shape: (99441, 10)
Unique customers: 99441


In [17]:
# Start with orders as the base table
ml_table = orders.merge(
    customers_enriched,
    on="customer_id",
    how="left"
)

# Add aggregated order-item information
ml_table = ml_table.merge(
    order_items_agg,
    on="order_id",
    how="left"
)

# Add aggregated payment information
ml_table = ml_table.merge(
    payments_agg,
    on="order_id",
    how="left"
)

print("ML table shape:", ml_table.shape)
print("Unique orders:", ml_table["order_id"].nunique())

ML table shape: (99441, 32)
Unique orders: 99441


In [18]:
# Save the final ML table for the EDA notebook
ml_table.to_pickle("ml_table.pkl")

print("Saved successfully.")
print("Shape:", ml_table.shape)

Saved successfully.
Shape: (99441, 32)


In [ ]:
print("Total rows:", len(ml_table))
print("Unique order IDs:", ml_table["order_id"].nunique())
print("Duplicate order IDs:", ml_table["order_id"].duplicated().sum())

Total rows: 99441
Unique order IDs: 99441
Duplicate order IDs: 0


In [19]:
print(ml_table.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'geolocation_zip_code_prefix', 'latitude', 'longitude', 'city', 'state', 'item_count', 'unique_products', 'unique_sellers', 'total_item_price', 'mean_item_price', 'max_item_price', 'total_freight_value', 'mean_freight_value', 'mean_product_weight_g', 'product_category_count', 'seller_state_count', 'payment_count', 'total_payment_value', 'mean_payment_value', 'max_payment_installments']


In [20]:
ml_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,max_item_price,total_freight_value,mean_freight_value,mean_product_weight_g,product_category_count,seller_state_count,payment_count,total_payment_value,mean_payment_value,max_payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,8.72,500.0,1.0,1.0,3.0,38.71,12.903333,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,22.76,400.0,1.0,1.0,1.0,141.46,141.460000,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,19.22,420.0,1.0,1.0,1.0,179.12,179.120000,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,27.20,450.0,1.0,1.0,1.0,72.20,72.200000,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,8.72,250.0,1.0,1.0,1.0,28.62,28.620000,1.0


In [21]:
import os

os.makedirs("artifacts", exist_ok=True)

ml_table.to_csv(
    "artifacts/order_level_ml_table.csv",
    index=False
)

print("Artifact saved successfully!")

Artifact saved successfully!


In [22]:
print("Final ML table shape:", ml_table.shape)
print("Unique orders:", ml_table["order_id"].nunique())
print("Duplicate order IDs:", ml_table["order_id"].duplicated().sum())

Final ML table shape: (99441, 32)
Unique orders: 99441
Duplicate order IDs: 0


In [23]:
ml_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,max_item_price,total_freight_value,mean_freight_value,mean_product_weight_g,product_category_count,seller_state_count,payment_count,total_payment_value,mean_payment_value,max_payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,8.72,500.0,1.0,1.0,3.0,38.71,12.903333,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,22.76,400.0,1.0,1.0,1.0,141.46,141.460000,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,19.22,420.0,1.0,1.0,1.0,179.12,179.120000,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,27.20,450.0,1.0,1.0,1.0,72.20,72.200000,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,8.72,250.0,1.0,1.0,1.0,28.62,28.620000,1.0
